# Chest BPM — private E4B QLoRA seed pilot
Keep this notebook **Private / Only you**. User-authorized Kaggle GPU route; no paid compute or public publishing. This notebook embeds only allowlisted app source and synthetic train/validation data. **No sealed final test, secrets, private chats/audio/member data or provider listings.**

Enable Internet and a free T4/L4 GPU in Session options before running. Only device 0 is used. It preflights resources, installs an isolated pinned environment, then runs one bounded pilot. Nothing changes the working Mac demo. Actual GPU/package compatibility is not yet verified.

Save Version → Save & Run All once for a fresh durable run, or run cells interactively and Quick Save **with outputs** afterward. Do not rerun training merely to save outputs. Download the final artifacts ZIP and confirm saved outputs before stopping the GPU session. A saved adapter alone is not proof of held-out improvement.


In [ ]:
import base64, hashlib, json, subprocess, sys, zipfile, io
from pathlib import Path, PurePosixPath
assert Path('/kaggle/working').is_dir(), 'Run this imported private notebook on Kaggle, not on the Mac'
BUNDLE_SHA256 = '9c4e9066c6d33fc4759b6f65d47531c4093d78cf3c52893033a8256927032d5a'
BUNDLE_BASE64 = '...'
ALLOWED = ['care-assistant/finetuning/EXPORT.md', 'care-assistant/finetuning/README.md', 'care-assistant/finetuning/config/baseline-lock.json', 'care-assistant/finetuning/config/candidate-package-pins.json', 'care-assistant/finetuning/config/kaggle-upstream-check.json', 'care-assistant/finetuning/config/task-contract.json', 'care-assistant/finetuning/config/train.json', 'care-assistant/finetuning/config/upstream-pins.json', 'care-assistant/finetuning/data/compiled.train.jsonl', 'care-assistant/finetuning/data/compiled.validation.jsonl', 'care-assistant/finetuning/data/seed.validation.jsonl', 'care-assistant/finetuning/data/tools.validation.jsonl', 'care-assistant/finetuning/kaggle/README.md', 'care-assistant/finetuning/kaggle/runner.py', 'care-assistant/finetuning/reports/data-validation.json', 'care-assistant/finetuning/requirements.candidate.txt', 'care-assistant/finetuning/scripts/bundle.py', 'care-assistant/finetuning/scripts/evaluate.mjs', 'care-assistant/finetuning/scripts/lock-environment.py', 'care-assistant/finetuning/scripts/runtime-contract.mjs', 'care-assistant/finetuning/scripts/train.py', 'care-assistant/package-lock.json', 'care-assistant/package.json', 'care-assistant/public/app.js', 'care-assistant/public/index.html', 'care-assistant/public/style.css', 'care-assistant/scripts/model-artifacts.js', 'care-assistant/scripts/start-local-model.js', 'care-assistant/src/conversation.js', 'care-assistant/src/directories.js', 'care-assistant/src/healthplan.js', 'care-assistant/src/knowledge.js', 'care-assistant/src/mcp.js', 'care-assistant/src/model.js', 'care-assistant/src/schema.js', 'care-assistant/src/server.js', 'care-assistant/src/tool-client.js', 'care-assistant/src/tools.js']
raw = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, 'Embedded bundle checksum mismatch'
workspace = Path('/kaggle/working/care-assistant-pilot')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert sorted(z.namelist()) == ALLOWED
    for name in z.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and '..' not in path.parts
        assert not any(s in name for s in ['.env', 'sealed-test', 'compiled.test', 'tools.test', 'evaluation/', '.cache/', 'node_modules/'])
    z.extractall(workspace)
app = workspace / 'care-assistant'
runner = app / 'finetuning/kaggle/runner.py'
assert runner.is_file()
print('Verified allowlisted bundle:', BUNDLE_SHA256, 'files:', len(ALLOWED))
print('Only synthetic train/validation are present. Final test content is absent.')
def stage(name):
    subprocess.run([sys.executable, str(runner), name], check=True)


## 1. Preflight — no package/model download
Checks Internet/public model metadata, assigned GPU, memory, driver and scratch/output headroom. A failure saves a no-success status. Fix the actual environment/access issue before continuing; do not switch to paid compute.


In [ ]:
stage('preflight')


## 2. Isolated environment
Installs pinned CUDA/training packages under `/tmp`, checks dependencies/imports/CUDA and saves the resolved environment lock. Does not modify Kaggle's preinstalled Python environment. The large model cache also stays in scratch rather than saved outputs. Stop and inspect logs if it fails.


In [ ]:
stage('setup')


## 3. One authorized bounded training run
This downloads the pinned public 4-bit base on Kaggle, verifies native template/assistant masks, trains text LoRA only, and saves the adapter plus tensor-change evidence. Train/validation only: sealed test content is absent. Existing run directories are not overwritten. There is no publishing, full-model export or default-model swap.


In [ ]:
stage('train')


## 4. Save and retrieve actual outputs
Download `/kaggle/working/care-e4b-pilot-artifacts.zip`. Inspect `training-complete.json` and adapter hashes. Successful weight updates are **not** a held-out/GGUF quality result. The matched-base/adapted llama.cpp b12248 evaluation described in EXPORT.md remains required before any adoption.

Confirm the private saved notebook version includes output files, then stop its GPU session. If stages fail, inspect the saved status/log archive; do not claim a trained adapter. Use `stage('package')` only to repackage existing outputs, not to retrain.


In [ ]:
from IPython.display import FileLink, display
output = Path('/kaggle/working/care-e4b-pilot-artifacts.zip')
assert output.is_file(), 'No output archive: inspect failed stage/logs'
display(FileLink(str(output)))
